# Analyzing Customer Behavior for E-commerce Insights

**Applicant:** Fianko Junior Owusu  
**Role:** Intelligent Systems Services Engineer  
**Organization:** Npontu Technologies Ltd

## Objective

This project analyses synthetic customer activity from an e-commerce platform. It cleans imperfect data, creates useful customer-behaviour features, explores business patterns, demonstrates partitioned processing with Dask, and trains a model to predict customer churn.

**Business question:** Which customers are likely to stop purchasing, and what actions can the company take to retain them?

## 1. Import the analysis functions

The complete, commented implementation is stored in `analysis.py`. Keeping reusable functions outside the notebook makes the project easier to test, maintain, and deploy.

In [ ]:
from analysis import (
    generate_synthetic_data,
    clean_and_engineer,
    perform_eda,
    run_dask_analysis,
    train_churn_model,
    create_business_recommendations,
)
from IPython.display import Image, display
import pandas as pd
pd.set_option("display.max_columns", 50)

## 2. Generate the synthetic dataset

The generated data represents customer demographics, browsing behaviour, purchase history, product interactions, support activity, and churn. Missing values, invalid entries, and duplicates are deliberately added so the cleaning process can be demonstrated.

In [ ]:
raw_df = generate_synthetic_data(n_customers=12000)
print(f"Dataset shape: {raw_df.shape}")
raw_df.head()

In [ ]:
raw_df.info()
raw_df.describe(include="all").T

## 3. Data cleaning and feature engineering

The cleaning stage removes duplicate customer IDs, replaces impossible ages and negative spending with missing values, and imputes numeric missing values using the median.

New features include customer tenure, purchase recency, average order value, cart-abandonment rate, monthly purchase frequency, engagement score, and age group. Numeric standardisation and categorical encoding are performed later inside the modelling pipeline to prevent data leakage.

In [ ]:
clean_df, quality_report = clean_and_engineer(raw_df)
quality_report

In [ ]:
clean_df[[
    "customer_id", "recency_days", "tenure_days", "average_order_value",
    "cart_abandonment_rate", "purchase_frequency", "engagement_score", "churn"
]].head()

## 4. Exploratory data analysis

The analysis compares churn, customer activity, spending, membership, device type, and location. The charts are also saved as high-resolution PNG files in `outputs/charts`.

In [ ]:
eda_insights = perform_eda(clean_df)
eda_insights

In [ ]:
display(Image(filename="outputs/charts/customer_insights.png"))
display(Image(filename="outputs/charts/correlation_heatmap.png"))

## 5. Big-data tool utilisation: Dask

Dask was selected because it provides a familiar pandas-like interface while splitting data into partitions that can be processed in parallel. Here, eight partitions simulate a distributed workflow. The approach can later scale to data larger than one machine's memory or run on a multi-machine Dask cluster.

In [ ]:
dask_city_analysis = run_dask_analysis(clean_df)
dask_city_analysis

## 6. Predictive modelling

A Random Forest classifier predicts whether a customer will churn. The preprocessing pipeline:

- imputes numeric and categorical values;
- standardises numeric features;
- one-hot encodes categorical features;
- prevents leakage by fitting transformations only on training data.

Performance is measured using accuracy, precision, recall, F1-score, ROC-AUC, a confusion matrix, and stratified five-fold cross-validation. F1 and ROC-AUC are important because churn classes may be imbalanced.

In [ ]:
model, model_metrics, feature_importance = train_churn_model(clean_df)
model_metrics

In [ ]:
display(Image(filename="outputs/charts/model_evaluation.png"))
display(Image(filename="outputs/charts/feature_importance.png"))
feature_importance.head(12)

## 7. Actionable business recommendations

The following recommendations combine the exploratory findings with the model's most important churn drivers.

In [ ]:
recommendations = create_business_recommendations(clean_df, feature_importance)
for number, recommendation in enumerate(recommendations, 1):
    print(f"{number}. {recommendation}")

## 8. Conclusion and limitations

This workflow shows how customer behaviour can be converted into churn risk and useful retention actions. The model is suitable as a proof of concept, not as a final production system, because its training data is synthetic.

Before production deployment, the business should retrain the model on consented real customer data, check fairness across customer groups, monitor performance and data drift, protect personal information, and periodically review whether retention actions actually reduce churn.